IMPORTANT

If you haven't already, create a .venv and install dependencies

After creating a .venv, run this in your code editor's terminal inside the .venv source location: "python -m pip install pandas ipykernel numpy jupyter"

In [1]:
import pandas as pd
import numpy as np

In [2]:
play_attention_scores = pd.read_csv("../outputs_csv/play_attention_scores.csv")
pffScoutingData = pd.read_csv("../cleaned_csv/pffScoutingData_cleaned.csv")

# The each_pass_rusher dataframe contains each pass rusher for each play.
each_pass_rusher = pd.read_csv("../outputs_csv/each_pass_rusher.csv")

**VERY IMPORTANT!**

The base_filtered file below is not in the GitHub because it is too large. It is in a .zip file in the shared **Google Drive** folder inside the "Output Datasets" folder. The zip file is called "Datasets Not in Github (Michael).zip". 

After extracting the .zip file, drag the base_filtered.csv file into the outputs_csv folder.

In [3]:
# The base_filtered dataframe contains frame-by-frame data of all players on only relevant plays.
base_filtered = pd.read_csv("../outputs_csv/base_filtered.csv")



/var/folders/rj/tzk4w23s0md7pcnpq4w2fy6c0000gn/T/ipykernel_27400/3775231977.py:2: DtypeWarning: Columns (38,39) have mixed types. Specify dtype option on import or set low_memory=False.
  base_filtered = pd.read_csv("../outputs_csv/base_filtered.csv")


In [4]:
pffScoutingData

,gameId,playId,nflId,pff_role,pff_positionLinedUp,pff_nflIdBlockedPlayer,pff_blockType,pff_backFieldBlock
0,2021090900,97,25511,Pass,QB,NaN,NaN,NaN
1,2021090900,97,35481,Pass Route,TE-L,NaN,NaN,NaN
2,2021090900,97,35634,Pass Route,LWR,NaN,NaN,NaN
3,2021090900,97,39985,Pass Route,HB-R,NaN,NaN,NaN
4,2021090900,97,40151,Pass Block,C,44955.0,SW,0.0
...,...,...,...,...,...,...,...,...
188249,2021110100,4433,52507,Pass Block,LT,43338.0,PP,0.0
188250,2021110100,4433,52546,Coverage,SCBoR,NaN,NaN,NaN
188251,2021110100,4433,52573,Pass Route,SLoWR,NaN,NaN,NaN
188252,2021110100,4433,52585,Pass Rush,LEO,NaN,NaN,NaN


In [5]:
# ball_snap_frames contains only frames when the ball is snapped
ball_snap_frames = base_filtered[base_filtered['event'] == 'ball_snap']

# base_filtered_pass_rushers is for pass rushers only, and it includes frames before the pass rush window unlike pass_rushers.
base_filtered_pass_rushers = base_filtered.merge(each_pass_rusher[['gameId', 'playId', 'nflId']].drop_duplicates(), on=['gameId', 'playId', 'nflId'], how='inner')

Feature-engineer variables for how far the pass rusher is from the center at the ball_snap instance (x distance, y distance, and total distance).

In [6]:
# filter ball_snap_frames to only include ball snap frames of pass rushers in each_pass_rusher df
ball_snaps_pass_rushers = ball_snap_frames.merge(each_pass_rusher[['gameId', 'playId', 'nflId']].drop_duplicates(), on=['gameId', 'playId', 'nflId'], how='inner')
ball_snaps_pass_rushers

# ball_snap_frames to only include centers
ball_snaps_centers = ball_snap_frames[ball_snap_frames['pff_positionLinedUp'] == 'C']
ball_snaps_centers

,gameId,playId,season,week,gameDate,quarter,down,yardsToGo,gameClock,play time,...,playDirection,x,y,s,a,dis,o,dir,event,frameIdEndWindow
149,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:31.600,...,right,42.10,24.02,0.51,1.98,0.06,48.28,308.34,ball_snap,36
1418,2021090900,137,2021,1,09/09/2021,1,1,10,13:18,28:10.500,...,left,109.23,24.06,0.92,0.88,0.09,279.04,170.92,ball_snap,31
2046,2021090900,187,2021,1,09/09/2021,1,2,6,12:23,29:15.500,...,left,77.67,26.46,0.54,1.41,0.06,284.80,121.34,ball_snap,27
2829,2021090900,282,2021,1,09/09/2021,1,1,10,9:56,31:52.100,...,left,49.82,30.07,0.58,0.87,0.06,265.80,123.15,ball_snap,36
3506,2021090900,349,2021,1,09/09/2021,1,3,15,9:46,34:05.600,...,left,55.01,30.11,0.82,2.07,0.08,276.68,94.61,ball_snap,32
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5163669,2021110100,4310,2021,8,11/01/2021,4,3,8,1:56,15:52.900,...,left,18.39,30.05,0.39,1.38,0.03,282.30,91.81,ball_snap,37
5164113,2021110100,4363,2021,8,11/01/2021,4,1,10,1:07,18:41.000,...,right,34.26,29.92,0.55,1.39,0.05,75.80,288.03,ball_snap,37
5164927,2021110100,4392,2021,8,11/01/2021,4,2,7,1:01,19:17.900,...,right,37.25,23.77,0.54,1.54,0.06,79.79,278.44,ball_snap,37
5165696,2021110100,4411,2021,8,11/01/2021,4,3,15,0:39,19:40.200,...,right,29.07,23.78,0.62,1.08,0.06,81.31,273.62,ball_snap,32


In [7]:
# calculate the difference in x, the difference in y, and the euclidean distance of each pass rusher to the center at the moment of the ball snap on each play.
temp = ball_snaps_pass_rushers.merge(ball_snaps_centers[['gameId', 'playId', 'x', 'y']], on=['gameId', 'playId'], how='inner', suffixes=('', '_center'))
temp['diff_x'] = temp['x'] - temp['x_center']
temp['diff_y'] = temp['y'] - temp['y_center']
temp['euclidean_distance'] = np.sqrt(temp['diff_x']**2 + temp['diff_y']**2)
temp

,gameId,playId,season,week,gameDate,quarter,down,yardsToGo,gameClock,play time,...,dis,o,dir,event,frameIdEndWindow,x_center,y_center,diff_x,diff_y,euclidean_distance
0,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:31.600,...,0.11,316.48,288.76,ball_snap,36,42.10,24.02,1.20,-5.13,5.268482
1,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:31.600,...,0.06,278.77,247.75,ball_snap,36,42.10,24.02,1.80,8.61,8.796141
2,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:31.600,...,0.06,243.27,288.42,ball_snap,36,42.10,24.02,1.25,1.16,1.705315
3,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:31.600,...,0.02,277.16,316.78,ball_snap,36,42.10,24.02,1.58,-2.09,2.620019
4,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:31.600,...,0.01,31.86,345.84,ball_snap,36,42.10,24.02,1.60,2.65,3.095561
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30966,2021110100,4411,2021,8,11/01/2021,4,3,15,0:39,19:40.200,...,0.04,286.55,249.67,ball_snap,32,29.07,23.78,1.22,4.86,5.010788
30967,2021110100,4433,2021,8,11/01/2021,4,4,15,0:35,20:21.800,...,0.06,250.03,271.87,ball_snap,37,29.15,23.72,1.16,6.07,6.179846
30968,2021110100,4433,2021,8,11/01/2021,4,4,15,0:35,20:21.800,...,0.03,250.01,251.87,ball_snap,37,29.15,23.72,1.04,-2.90,3.080844
30969,2021110100,4433,2021,8,11/01/2021,4,4,15,0:35,20:21.800,...,0.07,261.06,283.80,ball_snap,37,29.15,23.72,1.30,2.93,3.205448


In [8]:
# Run a multiple regression model. The three independent variables are diff_x, diff_y, and euclidean_distance in temp. 
# The dependent variable is avg_attention_score in play_attention_scores. 

training_dataset = temp.merge(play_attention_scores.rename(columns={'rusher_nflId': 'nflId'}), on=['gameId', 'playId', 'nflId'], how='inner')
X = training_dataset[['diff_x','diff_y', 'euclidean_distance']]
y = training_dataset['avg_attention_score']
from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit(X, y)
print("Coefficients:", model.coef_)
print("Intercept:", model.intercept_)

# r2score
from sklearn.metrics import r2_score
y_pred = model.predict(X)
print("R^2 Score:", r2_score(y, y_pred))


Python(27653) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Coefficients: [ 0.00154197  0.00039535 -0.14094692]
Intercept: 1.7347556649748577
R^2 Score: 0.2990456656993985


In [9]:
# Run a linear regression model with only euclidean_distance.
X = training_dataset[['euclidean_distance']]
y = training_dataset['avg_attention_score']
from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit(X, y)
print("Coefficients:", model.coef_)
print("Intercept:", model.intercept_)

# r2score
from sklearn.metrics import r2_score
y_pred = model.predict(X)
print("R^2 Score:", r2_score(y, y_pred))

Coefficients: [-0.14095643]
Intercept: 1.734694984473582
R^2 Score: 0.2990114739504528


In [10]:
# Calculate gravity for Myles Garrett based on this regression model.
myles_garrett_temp = training_dataset[training_dataset['displayName'] == 'Myles Garrett']
myles_garrett_temp['predicted_attention_score'] = model.predict(myles_garrett_temp[['euclidean_distance']])
myles_garrett_temp['gravity_score'] = myles_garrett_temp['avg_attention_score'] - myles_garrett_temp['predicted_attention_score']
myles_garrett_avg_gravity = myles_garrett_temp['gravity_score'].mean()
print("Myles Garrett's average gravity:", myles_garrett_avg_gravity)  

# Calculate gravity for Aaron Donald based on this regression model.
aaron_donald_temp = training_dataset[training_dataset['displayName'] == 'Aaron Donald']
aaron_donald_temp['predicted_attention_score'] = model.predict(aaron_donald_temp[['euclidean_distance']])
aaron_donald_temp['gravity_score'] = aaron_donald_temp['avg_attention_score'] - aaron_donald_temp['predicted_attention_score']
aaron_donald_avg_gravity = aaron_donald_temp['gravity_score'].mean()
print("Aaron Donald's average gravity:", aaron_donald_avg_gravity)  

# Calculate gravity for Justin Hollins based on this regression model.
justin_hollins_temp = training_dataset[training_dataset['displayName'] == 'Justin Hollins']
justin_hollins_temp['predicted_attention_score'] = model.predict(justin_hollins_temp[['euclidean_distance']])
justin_hollins_temp['gravity_score'] = justin_hollins_temp['avg_attention_score'] - justin_hollins_temp['predicted_attention_score']
justin_hollins_avg_gravity = justin_hollins_temp['gravity_score'].mean()
print("Justin Hollins's average gravity:", justin_hollins_avg_gravity)


Myles Garrett's average gravity: 0.1531803771832703
Aaron Donald's average gravity: 0.3501976731976896
Justin Hollins's average gravity: -0.10336685784698582


/var/folders/rj/tzk4w23s0md7pcnpq4w2fy6c0000gn/T/ipykernel_27400/3815947818.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  myles_garrett_temp['predicted_attention_score'] = model.predict(myles_garrett_temp[['euclidean_distance']])
/var/folders/rj/tzk4w23s0md7pcnpq4w2fy6c0000gn/T/ipykernel_27400/3815947818.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  myles_garrett_temp['gravity_score'] = myles_garrett_temp['avg_attention_score'] - myles_garrett_temp['predicted_attention_score']
/var/folders/rj/tz